# astrovodka 使用教程 — 从 TNG 子晕到合成 HI 数据立方体

本教程带你完整走一遍 astrovodka 的工作流，**每一步都说明为什么这么做**，
以及哪里容易踩坑。全部内容可以在 TNG-100 的 snapshot 99 上直接跑通。

### 你将学会

| 章节 | 内容 |
|---|---|
| [1](#1-astrovodka-是什么) | astrovodka 是什么、和 MARTINI 的关系 |
| [2](#2-环境准备) | 环境准备 |
| [3](#3-配置) | 子晕怎么选、两种红移、两种距离 |
| [4](#4-加载源与-hi-模型) | 四种 HI 模型、怎么切换、结果怎么读 |
| [5](#5-构建合成观测) | DataCube / beam / noise / 光谱模型，以及验证 |
| [6](#6-三维数据产品) | 3D 网格产物（UV 场、DM/HI/H₂ 密度） |
| [7](#7-可视化数据立方体) | 数据立方体的可视化 |
| [8](#8-一个必须知道的陷阱martini-会就地裁剪-source) | `Martini` 会就地裁剪 `source` —— 放在最后是因为它的演示会破坏 source |
| [9](#9-疑难排解) | 常见报错与陷阱 |
| [10](#10-小结与下一步) | 小结与下一步 |

### 前置条件

- **TNG-100 数据**：本教程用 snapshot 099。数据路径通过环境变量
  `TNG_BASE_PATH` 传入，不写死在 notebook 里：
  ```bash
  export TNG_BASE_PATH=/path/to/TNG/TNG-100/
  ```
- **Python 环境**：需要 `numpy`、`scipy`、`astropy`、`h5py`、`matplotlib`。
  安装方式见 [安装指南](../../docs/zh/installation.md)。

> **本教程的 notebook 不保存输出。** 每节的数字都会在运行时打印，
> 所以你机器上的结果会和这里描述的一致，但不依赖任何写死的数值。


---

## 1. astrovodka 是什么

astrovodka 把宇宙学流体模拟（这里是 IllustrisTNG）里的**气体粒子**，
变成**射电望远镜会看到的东西** —— 带噪声和波束的 21 cm 数据立方体。

它 fork 自 [MARTINI](https://github.com/kyleaoman/martini)（PyPI 上的 `astromartini`），
针对高红移工作改了两件事。

### 1.1 管线

```
                    ┌─────────────────────────────────────────────┐
   TNG 快照文件 ───▶ │  TNGSource                                   │
   (groups_099,     │   · 读气体/恒星/DM 粒子                       │
    snapdir_099)    │   · 用选定的 HI 模型算逐粒子 HI/H₂ 质量  ◀── himodel
                    │   · 用 Planck15 算宇宙学距离                  │
                    └───────────────────┬─────────────────────────┘
                                        │  mHI_g, 位置, 速度, 温度, 平滑长度
                                        ▼
                    ┌─────────────────────────────────────────────┐
                    │  Martini   （观测流程编排）                    │
                    │   insert_source_in_cube()  逐像素铺谱线        │
                    │   add_noise()              加噪声             │
                    │   convolve_beam()          卷波束             │
                    │   write_fits() / write_hdf5()                │
                    └───────────────────┬─────────────────────────┘
                                        ▼
                              DataCube ──▶ FITS 立方体
                    （像素/通道网格 + WCS 天球坐标）
```

模块分工：

| 模块 | 职责 |
|---|---|
| `astrovodka.himodel` | **HI/H₂ 物理**（本包自己的，四种模型） |
| `astrovodka.sources.TNGSource` | 读 TNG 数据、装配源 |
| `astrovodka.DataCube` | 输出网格与 WCS |
| `astrovodka.Martini` | 铺谱线 / 加噪声 / 卷波束 / 写文件 |
| `astrovodka.beams`、`noise`、`spectral_models`、`sph_kernels` | 可替换的物理组件 |

`himodel` 和观测层是**单向依赖**：物理层不认识数据立方体，
所以它可以脱离整个观测流程单独使用（第 4.2 节就是这么做的）。


### 1.2 四种 HI 模型

这是 astrovodka 相对 MARTINI 最主要的改动。MARTINI 用 Leroy+2008 的
气压定标把气体分成 HI 和 H₂，那是近邻星系的定标、**没有红移依赖**，
用在高红移上不合适。

astrovodka 实现了 Diemer+2018（arXiv:1806.02341）的四种方法，
即 {Gnedin & Draine 2014, Krumholz+ 2013} × {体积版, 投影版}：

| `hi_model` | 面密度从哪来 | 出处 |
|---|---|---|
| `GD14_vol` | 逐单元 Jeans 近似，Σ = λ_J ρ | GD14 附录 C.3 |
| `GD14_map` | 面向观测者的 128² 投影 | 同上 + §2.3 |
| **`K13_vol`**（默认） | 逐单元 Jeans 近似 | K13 附录 C.4 |
| `K13_map` | 面向观测者的 128² 投影 | 同上 + §2.3 |

**体积版**：每个气体单元用自己的密度和内能算出一个 Jeans 长度，
把体密度换成面密度，再算分子分数。

**投影版**：先把星系投影成一张面向观测者的 2D 图，每个单元取**它所在像素**的
分子分数。它们需要恒星粒子（要算恒星的表面密度和速度弥散），
而且**投影盒外的气体 HI 质量记为零** —— 模型定义的"星系"就是那个盒子。

### 怎么选？

| 你的目的 | 用哪个 |
|---|---|
| 默认、最省事、结果可对照已发表工作 | `K13_vol` |
| 要看模型系统性差异（论文里常做的对比） | 四个都跑，见第 4.2 节 |
| 想避免 Jeans 近似、直接测面密度 | `*_map` |
| 源很延展、不想丢外围气体 | 体积版；或给 map 版设 `map_clip_outside=True` |

Diemer+2018 对标官方 catalog 的实测偏差（前 200 个子晕）：

| 方法 | z=0 | z=0.5 | z=1.0 |
|---|---|---|---|
| GD14 vol | +4.0% | +3.0% | +1.9% |
| GD14 map | −5.0% | −10.6% | −12.2% |
| K13 vol | −8.0% | −0.3% | +1.6% |
| K13 map | −1.6% | −1.6% | −1.9% |


### 1.3 相对 MARTINI 的改动

| 方面 | MARTINI | astrovodka |
|---|---|---|
| 距离 | 线性哈勃定律 `v = H₀·D` | `astropy.cosmology.Planck15`：D_L 管流量，D_A 管角尺度 |
| 系统速度 | `h·100·D` | 射电惯例 `c·z/(1+z)`，与 WCS 谱轴一致 |
| HI 模型 | 气压定标（Leroy+2008） | 四种可选 Diemer+2018 模型 |
| UV 场 | — | 逐星系自适应 FFT 卷积，以 UVB 为下限 |
| 数据读取 | 网络 API + JupyterLab | **本地磁盘**（离线、快） |
| 子晕加载 | 整个 FOF 群 | 可选仅子晕（默认，快得多） |
| 定心 | catalog 的 `SubhaloPos` | **DM 质心** |
| H₂ | 不提供 | `source.m_H2_g`、`source.f_mol_g` |
| 三维产物 | — | `SubhaloDataProducts` |

> **没有任何 `import martini`。** MARTINI 的观测机器（数据立方体、波束、
> 光谱模型、SPH 核、噪声）是**摘进本仓库**的，不是外部依赖。
> `tests/test_selfcontained.py` 用 AST 解析强制这一点。


---

## 2. 环境准备

In [ ]:
import os
import sys
import warnings
from pathlib import Path


def find_repo_root():
    """Locate the checkout, so the tutorial runs from any working directory."""
    override = os.environ.get('ASTROVODKA_ROOT')
    if override:
        return Path(override).resolve()
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'pyproject.toml').is_file() and (parent / 'astrovodka').is_dir():
            return parent
    return None


REPO_ROOT = find_repo_root()

# Use an installed astrovodka if there is one (pip install -e .); otherwise fall
# back to importing it straight from the checkout found above.
try:
    import astrovodka  # noqa: F401
except ModuleNotFoundError:
    if REPO_ROOT is None:
        raise ModuleNotFoundError(
            'astrovodka is not installed and no checkout was found. Run '
            '`pip install -e .` from the repository root, or set ASTROVODKA_ROOT '
            'to its path.'
        ) from None
    sys.path.insert(0, str(REPO_ROOT))

import h5py
import numpy as np
import astropy.units as U
from astropy.cosmology import Planck15
from astropy.io import fits
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle

# ===== astrovodka =====
from astrovodka import (Martini, DataCube, SubhaloDataProducts,
                        GroupDataProducts)
from astrovodka.group_products import ism_radius
from astrovodka.sources import TNGSource
from astrovodka.beams import GaussianBeam
from astrovodka.noise import GaussianNoise
from astrovodka.spectral_models import GaussianSpectrum
from astrovodka.sph_kernels import CubicSplineKernel
from astrovodka.cosmology_utils import (
    luminosity_distance, angular_diameter_distance, radio_velocity_from_redshift,
)
from astrovodka.himodel import (
    METHODS, DEFAULT_METHOD, HIModelConfig, calc_UMW_global_Faucher11,
    compute_hi_mass,
)

# 物理层的底层入口，第 4.2 节对比四种模型时用
from astrovodka.sources._illustris_tools import (
    loadHeader, loadSingle, loadSubset, getSnapOffsets,
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150})

# 图里的中文标签需要 CJK 字体，否则 matplotlib 会画成方框（tofu）。
# 按可用性挑一个，挑不到就退回默认（图上的中文会显示成方框，但不影响计算）。
import matplotlib
_available = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _cand in ('Noto Sans CJK JP', 'Noto Sans CJK SC', 'Source Han Sans SC',
              'WenQuanYi Zen Hei', 'Droid Sans Fallback'):
    if _cand in _available:
        plt.rcParams['font.sans-serif'] = [_cand] + list(plt.rcParams['font.sans-serif'])
        print(f'CJK 字体: {_cand}')
        break
else:
    print('警告：没找到 CJK 字体，图上的中文会显示成方框')
plt.rcParams['axes.unicode_minus'] = False   # 负号也走同一套字体

print('astrovodka', __import__('astrovodka').__version__)
print('numpy', np.__version__, '| astropy', U.__name__ and __import__('astropy').__version__)
print('可用的 HI 模型:', METHODS, '| 默认:', DEFAULT_METHOD)

---

## 3. 配置

### 3.1 怎么选子晕（第一个大坑）

**TNG 的子晕 catalog 是按质量降序排的**，所以 ID 越小质量越大：

- **ID 0 是团核心**，约 3800 万个气体单元，单次 `loadSubhalo` 峰值超过 10 GB；
- 想抽样验证时**不要用 `valid_ids[:N]`**，那取到的全是团核心；
- 要稀疏化请用固定 stride，或直接用 100~200 区间的 ID。

本教程用 **subhalo 113**。选它是因为演示需要 HI **又多、又在恒星形成**：

| | **113**（本教程） | 139 | 122 |
|---|---|---|---|
| 气体单元 | 8169 | 7246 | 4448 |
| M_HI [M☉] | 4.3e9 | **4.5e9** | 3.3e9 |
| M_H₂ [M☉] | **1.8e9** | 3.0e8 | 1.9e8 |
| SFR [M☉/yr] | 0.98 | 0.17 | 0.19 |
| UV 场峰值 / 本底 | **27.6 / 0.0014（≈19000×）** | 2.2 | 6.0 |
| `adaptive_fft` vs `global` 的 HI 差 | **+13.9%** | +5.7% | +4.3% |

139 的 HI 总量高 5%，但它几乎不造星，UV 场峰值只有本底的 1550 倍 ——
用它演示，自适应 FFT UV 场的效果基本看不出来。113 能看出 14% 的差别。

> 换子晕只改下面 `subID` 一个变量。跑得快的小星系（约 4000~9000 单元）
> 大多在 100~200 区间。

In [ ]:
# ===================== 配置 =====================
# TNG 数据路径从环境变量读，不写死在 notebook 里：
#     export TNG_BASE_PATH=/path/to/TNG/TNG-100/
basePath = os.environ.get('TNG_BASE_PATH', '')
if not basePath:
    raise RuntimeError(
        '请先把 TNG-100 数据目录告诉 notebook：\n'
        '    export TNG_BASE_PATH=/path/to/TNG/TNG-100/\n'
        '（目录里应当有 groups_099/ 与 snapdir_099/）'
    )
basePath = basePath.rstrip('/') + '/'

snapNum  = 99        # TNG-100 snapshot，z ≈ 0
subID    = 113       # 见上一节：HI 多且正在造星
Z_OBS    = 0.02      # 把模拟放在哪个观测红移（见 3.2 节）
HI_MODEL = 'K13_vol' # 四种之一：K13_vol / K13_map / GD14_vol / GD14_map

# 第 6.3 节：FOF group。subhalo 113 属于 group 0，也就是盒子里最大的星系团——
# 不适合在 notebook 里载入，所以换一个较小的 group 作演示。选法见 6.3。
GROUP_ID    = 59    # 第 6.3 节分析的 FOF group
GROUP_NGRID = 192   # group 产品的网格边长

# 产物写到 examples/outputs/（已在 .gitignore 中），路径相对仓库根目录解析。
OUTDIR = str((REPO_ROOT or Path.cwd()) / 'examples' / 'outputs') + '/'
os.makedirs(OUTDIR, exist_ok=True)

if not os.path.isdir(basePath + f'groups_{snapNum:03d}'):
    raise RuntimeError(
        f'{basePath} 下找不到 groups_{snapNum:03d}/，请检查 TNG_BASE_PATH。'
    )

cosmo = Planck15

# ---- 这个配置对应的几何 ----
DL = luminosity_distance(Z_OBS, cosmo)
DA = angular_diameter_distance(Z_OBS, cosmo)
arcsec_to_kpc = (DA.to(U.kpc) / U.rad).to(U.kpc / U.arcsec)

print(f'观测红移 z       = {Z_OBS}')
print(f'光度距离   D_L   = {DL:.1f}   ← 决定流量')
print(f'角直径距离 D_A   = {DA:.1f}   ← 决定角尺度')
print(f'尺度       1"    = {arcsec_to_kpc.value:.3f} kpc')
print(f'系统速度   vsys  = {radio_velocity_from_redshift(Z_OBS):.0f}')
print(f'输出目录         = {OUTDIR}')

### 3.2 快照红移 vs 观测红移

这是最容易搞混的地方，TNGSource 把它们**分开**：

| | 从哪来 | 干什么用 |
|---|---|---|
| **快照红移** `z_snapshot` | 快照 header | **物理**：单位换算、HI 模型。必须用模拟自己的红移，否则质量全错 |
| **观测红移** `observed_redshift` | 你传的参数 | **几何**：D_L、D_A、vsys。决定"把星系搬到多远去看" |

不传 `observed_redshift` 时两者相同 —— 对 snapshot 99（z≈0）意味着
D_A 几乎为零，星系在天空上缩成一个点，立方体里什么都没有。

传了 `observed_redshift=0.02`，就是**用一个 z≈0 星系的物理，放在
z=0.02 的距离上观测**。这是高红移研究的常用做法：
模拟本身够不到那么远，但探测器的角分辨率和灵敏度要求星系在某个距离上。

下面看看同一个星系放在不同距离上的样子。

**注意** `HI_MODEL`、`subID` 等变量在上一格定义，`source` 还没建；
这里只算几何，不碰数据。

In [ ]:
rows = []
v_snapshot = radio_velocity_from_redshift(2.2e-16)   # snapshot 99 的红移

for z in [0.0, 0.005, 0.02, 0.05, 0.1]:
    dl = luminosity_distance(z, cosmo)
    da = angular_diameter_distance(z, cosmo)
    # 同一颗星系放在不同距离上，在天球上的张角按 D_A 反比缩放
    rows.append((z, dl.value, da.value, (da / DA).value,
                 radio_velocity_from_redshift(z)))

print(f'{"z_obs":>7} {"D_L [Mpc]":>11} {"D_A [Mpc]":>11} '
      f'{"角尺度相对 z=0.02":>18} {"vsys [km/s]":>12}')
print('-' * 64)
for z, dl, da, rel, v in rows:
    print(f'{z:>7.3f} {dl:>11.1f} {da:>11.1f} {rel:>18.3f} {v:>12.0f}')

print(f'\n星系自身的 z ≈ 0，vsys 用射电惯例 c·z/(1+z) = {v_snapshot:.1f} km/s')
print('z=0 时 D_A 落到 ~4.4 kpc 的下限，astrovodka 会发 UserWarning 提醒你。')

### 3.3 两个距离，各管各的

这是 v0.2.0 修掉的一个大 bug（当时 D_A 算了却没用，所有角尺度小了 (1+z)²）。

| 用途 | 用哪个距离 | 为什么 |
|---|---|---|
| 天球位置、RA/Dec、角尺度、SPH 核的角宽度 | **D_A** | 角尺度 = 物理尺度 / D_A |
| 流量的归一化 | **D_L** | S ∝ M / D_L² |
| 系统速度 `vsys` | 射电惯例 `c·z/(1+z)` | 与 WCS 谱轴 `VRAD` 一致 |

流量那一条差 (1+z)⁴ —— 用错了立方体的总流量就不是这个星系该有的。
你不需要手动处理：`TNGSource` 已经把三者接好了，但读代码时要知道
`source.skycoords.distance` 是 **D_A**，而 `source.luminosity_distance` 是 **D_L**。

---

## 4. 加载源与 HI 模型

`TNGSource` 一个类干完四件事：读快照、算距离、算 HI 质量、把一切交给 `SPHSource`。

```python
source = TNGSource(
    basePath=..., snapNum=99, subID=113,
    hi_model='K13_vol',      # 四种模型任选
    hi_config=None,          # HIModelConfig 或 dict，见 4.3
    uv_mode='adaptive_fft',  # 体积版专用，见 4.4
    observed_redshift=0.02,  # 观测红移，见 3.2
    load_subhalo_only=True,  # True 只读子晕，False 读整个 FOF 群
    ra=30*U.deg, dec=45*U.deg,
)
```

`hi_model` 在**读盘之前**就校验，写错了不会白等一次几 GB 的加载。

> **`load_subhalo_only=False` 什么时候用？** 当目标星系的前景/背景气体
> 会落进立方体视场时。默认 `True` 只读这个子晕自己的粒子，快得多，
> 但对团里的卫星星系可能漏掉同群其他成员的气体。

In [ ]:
if HI_MODEL not in METHODS:
    raise ValueError(f'HI_MODEL 必须是 {METHODS} 之一，收到 {HI_MODEL!r}')

print(f'加载 TNG-100 snapshot {snapNum} subhalo {subID} ...')
source = TNGSource(
    basePath=basePath,
    snapNum=snapNum,
    subID=subID,
    hi_model=HI_MODEL,
    observed_redshift=Z_OBS,
    load_subhalo_only=True,
    ra=30.0 * U.deg,
    dec=45.0 * U.deg,
)

print(f'\n=== 源概况 ===')
print(f'  粒子数：气体 {source.n_gas}，恒星 {source.n_star}，DM {source.n_dm}')
print(f'  快照红移 {source.source_redshift:.6f}  →  观测红移 {Z_OBS}')
print(f'  D_L = {source.D_L:.1f}   D_A = {source.D_A:.1f}')
print(f'  vsys = {source.vsys.to(U.km/U.s):.0f}')
print(f'\n=== 质量预算 ===')
print(f'  HI   {source.m_HI_total:.4e}')
print(f'  H2   {source.m_H2_g.sum():.4e}')
print(f'  气体 {source.m_gas_total:.4e}')
print(f'  恒星 {source.m_star_total:.4e}')
print(f'  DM   {source.m_dm_total:.4e}')
print(f'  中性气体占气体总量 {100*(source.m_HI_total + source.m_H2_g.sum())/source.m_gas_total:.1f}%')
print(f'\n  用的模型：{source.hi_model}   ({source.hi_model_spec["reference"]})')
print(f'  模型配置：{source.hi_config.summary()}')
print(f'  是全投影模型吗：{source.is_projected}')

# ---- 源的角尺度与速度宽度：决定立方体该开多大 ----
xyz  = source.coordinates_g.get_xyz()
vxyz = source.coordinates_g.differentials['s'].get_d_xyz()

y_kpc = 1.1 * (xyz[1].max() - xyz[1].min()).to_value(U.kpc)
z_kpc = 1.1 * (xyz[2].max() - xyz[2].min()).to_value(U.kpc)
v_kms = 1.1 * max((vxyz[0].max() - vxyz[0].min()).to_value(U.km/U.s), 20.0)

y_extent_arcsec = y_kpc / arcsec_to_kpc.value
z_extent_arcsec = z_kpc / arcsec_to_kpc.value
print(f'\n=== 源的尺度（决定立方体视场，1.1× 粒子范围）===')
print(f'  天球 Y (RA)  {y_extent_arcsec:6.1f}"  = {y_kpc:6.1f} kpc')
print(f'  天球 Z (Dec) {z_extent_arcsec:6.1f}"  = {z_kpc:6.1f} kpc')
print(f'  视向速度展宽 {v_kms:6.0f} km/s')
print(f'\n  也就是说：视场至少要 {max(y_extent_arcsec, z_extent_arcsec):.0f}" 才能装下整个星系，')
print(f'  否则会被 Martini 剔除掉（第 5.4 节会看到这件事）。')

### 4.1 模型到底算出了什么

除了 `source.m_HI_total`，逐粒子的中间量也都留着 —— 调试和做物理解释时很有用：

| 属性 | 形状 | 含义 |
|---|---|---|
| `source.mHI_g` | (n_gas,) | 逐粒子 HI 质量 [M☉] |
| `source.m_H2_g` | (n_gas,) | 逐粒子 H₂ 质量 [M☉] |
| `source.f_mol_g` | (n_gas,) | 逐单元**分子比例** f_H₂/(f_HI+f_H₂) |
| `source.f_neutral_g` | (n_gas,) | 逐单元中性比例（恒星形成单元取 1） |
| `source.u_mw_g` | (n_gas,) | 逐单元 UV 场 [Draine 单位，1 = 银河系] |
| `source.hi_diagnostics` | dict | 模型自身的诊断（投影几何、对齐等） |
| `source.hi_result` | `HiMassResult` | 上面全部的容器，还有 `cells` 表 |

把分子比例和 UV 场画出来，能直接看出模型在做什么。

> 图里每个点是一个气体单元。恒星形成的单元（f_neutral = 1，图 (c) 里的黄色）
> 密度高、UV 场强，但它们的**分子比例也最高** —— 因为决定 f_mol 的是
> 面密度 τ_c 那一项，稠密气体哪怕被强 UV 照着也还是分子为主。
>
> 所以 (c) 这张图**看起来**是"UV 越强分子越多"，那是密度和 UV 在真实星系里
> 正相关造成的假象。模型里 UV 的**反**相关作用（K13 的 χ 项）只有在
> **固定密度**下才看得出来 —— `tests/test_hi_models.py` 里的
> `test_more_uv_means_fewer_molecules` 就是那么测的。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

u   = np.asarray(source.u_mw_g, dtype='float64')
fm  = np.asarray(source.f_mol_g, dtype='float64')
fne = np.asarray(source.f_neutral_g, dtype='float64')
sfr = np.asarray(source.sfr_g, dtype='float64')
uvb = calc_UMW_global_Faucher11(source.source_redshift)  # 真正的 UVB 本底

# (a) 逐单元 UV 场
ax = axes[0]
ax.hist(u, bins=np.logspace(np.log10(max(u.min(), 1e-8)), np.log10(u.max()), 50),
        color='C3', alpha=.85)
ax.set_xscale('log')
ax.axvline(uvb, color='k', ls='--', lw=1.2, label=f'UVB 本底 {uvb:.2e}')
ax.set_xlabel('U_MW  [Draine units]')
ax.set_ylabel('单元数')
ax.set_title('(a) 逐单元 UV 场')
ax.legend()

# (b) 分子比例分布
ax = axes[1]
ax.hist(fm, bins=50, color='C0', alpha=.85)
ax.set_yscale('log')
ax.set_xlabel(r'$f_\mathrm{mol} = f_{H_2}/(f_{HI}+f_{H_2})$')
ax.set_ylabel('单元数')
ax.set_title('(b) 分子比例分布')

# (c) f_mol vs U_MW，按中性比例着色
ax = axes[2]
sc = ax.scatter(u, fm, c=fne, s=7, alpha=.55, cmap='viridis')
ax.set_xscale('log')
ax.set_xlabel('U_MW  [Draine units]')
ax.set_ylabel(r'$f_\mathrm{mol}$')
ax.set_title('(c) f_mol vs UV 场')
plt.colorbar(sc, ax=ax, label=r'$f_\mathrm{neutral}$', shrink=.85)

for ax in axes:
    ax.grid(alpha=.2)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_model_diagnostics.png'),
            bbox_inches='tight', facecolor='white')
plt.show()

# H2 的质量加权占比（不是 f_mol 的简单平均）
h2_frac = source.m_H2_g.sum() / (source.m_HI_total + source.m_H2_g.sum())
print(f'UV 场        {u.min():.4g} .. {u.max():.4g}')
print(f'  UVB 本底   {uvb:.4g}   →  峰值是本底的 {u.max()/uvb:.0f} 倍')
print(f'  （本星系到处都超过本底，说明逐星系的 FFT 场确实在起作用）')
print(f'f_mol        中位 {np.median(fm):.4f}，最大 {fm.max():.4f}')
print(f'H2 质量占比  {100*h2_frac:.1f}%')
print(f'恒星形成单元 {int((sfr > 0).sum())} / {source.n_gas}')

### 4.2 切换模型：四种都跑一遍

`TNGSource` 每次构造都会重新读盘，所以对比四种模型时更省事的做法是
**只读一次数据，然后调用物理层四次**：

```python
from astrovodka.himodel import HIModelConfig, compute_hi_mass

cfg = HIModelConfig(h=..., z=...)          # h、z 来自快照
res = compute_hi_mass(gas, model='K13_map', cfg=cfg, stars=stars, sub=sub)

res.m_hi          # 逐单元，TNG code unit（1e10 M☉/h），长度 = 气体单元数
res.m_hi_msun     # 总量，M☉
res.diagnostics   # 投影几何、对齐、盒内外气体占比……
```

`compute_hi_mass` 的 `model` 大小写不敏感，`'K13'`/`'GD14'` 指体积版，
`('K13','map')` 这种二元组写法也接受。

> **这就是 `himodel` 单向依赖的好处**：物理层不需要数据立方体，
> 可以脱离整套观测流程单独调用、单独测试。

下面这一格的 `gas`/`stars`/`sub` 是直接从快照读的原始字段，
用 `TNGSource` 内部的同一套读取器 —— 这样四次计算共享同一份数据。

In [ ]:
# ---- 只读一次数据，四种模型共用 ----
hdr = loadHeader(basePath, snapNum)
h_sim, z_sim = float(hdr['HubbleParam']), float(hdr['Redshift'])
cfg = HIModelConfig(h=h_sim, z=z_sim, boxsize_ckpc_per_h=float(hdr['BoxSize']))

subset = getSnapOffsets(basePath, snapNum, subID, 'Subhalo')

gas_fields  = ['Masses', 'Velocities', 'InternalEnergy', 'ElectronAbundance',
               'NeutralHydrogenAbundance', 'StarFormationRate', 'Density',
               'Coordinates', 'CenterOfMass', 'GFM_Metals', 'GFM_Metallicity']
gas_mdi     = [None]*9 + [0] + [None]          # GFM_Metals 只要 H 那一列
star_fields = ['Masses', 'Coordinates', 'Velocities', 'GFM_StellarFormationTime']
gas_raw  = loadSubset(basePath, snapNum, 'gas',  fields=gas_fields,
                      subset=subset, mdi=gas_mdi)
star_raw = loadSubset(basePath, snapNum, 'star', fields=star_fields, subset=subset)

# 投影模型需要 subhalo 自己的位置、速度和半质量半径（原始 ckpc/h）
row   = loadSingle(basePath, snapNum, subhaloID=subID)
half  = np.asarray(row['SubhaloHalfmassRadType'], dtype='float64')
sub   = dict(pos=np.asarray(row['SubhaloPos'], dtype='float64'),
             vel=np.asarray(row['SubhaloVel'], dtype='float64'),
             rhalf_gas=float(half[0]), rhalf_star=float(half[4]))

print(f'读入 {gas_raw["count"]} 个气体单元、{star_raw["count"]} 个恒星粒子')
print(f'R½(gas) = {half[0]:.1f} ckpc/h,  R½(star) = {half[4]:.1f} ckpc/h\n')

# ---- 四种模型各跑一次 ----
results = {}
for m in METHODS:
    kw = dict(stars=star_raw, sub=sub) if m.endswith('_map') else {}
    results[m] = compute_hi_mass(gas_raw, m, cfg, **kw)

# ---- 对比表 ----
print(f'{"模型":<10} {"M_HI [M☉]":>13} {"M_H2 [M☉]":>13} {"M_中性 [M☉]":>13} '
      f'{"f_mol":>7} {"有效单元":>9}')
print('-' * 72)
for m, r in results.items():
    n_eff = int(np.count_nonzero(r.m_neutral))
    fmol = r.m_h2_msun / r.m_neutral_msun if r.m_neutral_msun else 0.0
    print(f'{m:<10} {r.m_hi_msun:>13.4e} {r.m_h2_msun:>13.4e} '
          f'{r.m_neutral_msun:>13.4e} {fmol:>7.3f} {n_eff:>9d}')

hi_vol = results['K13_vol'].m_hi_msun
print(f'\n以 K13_vol 为基准的 HI 偏差：')
for m, r in results.items():
    print(f'  {m:<10} {100*(r.m_hi_msun/hi_vol - 1):+6.1f}%')

print('\n注意两个体积版的**中性总量完全相同** —— 模型只在 HI/H2 之间怎么分上不同，')
print('不改变有多少中性氢。投影版少一点，是因为投影盒外的气体被记为零质量。')

### 4.3 调模型参数：`HIModelConfig`

`HIModelConfig` 是一个 frozen dataclass，装 `h`、`z` 和所有数值开关。

```python
cfg = HIModelConfig(h=0.6774, z=0.0, ngrid=128)
```

传进 `TNGSource` 时可以用 dict（只覆盖想改的），也可以用完整对象：

```python
TNGSource(..., hi_config={'ngrid': 64, 'k13_variant': 'hydro'})
```

> **`h` 和 `z` 永远以快照为准。** 你在 `hi_config` 里写的 `h`/`z` 会被
> 快照 header 的值覆盖，因为用别的快照的单位去解释这份数据是没有意义的，
> 静默出错又看不出来。实际生效的配置在 `source.hi_config`。

主要开关：

| 字段 | 默认 | 含义 |
|---|---|---|
| `ngrid` | 128 | 3D FFT UV 场网格边长 |
| `f_esc` | 0.1 | UV 逃逸分数（Diemer+2018 §4.2 的标定值） |
| `box_expand` | 1.1 | 自适应 UV 盒 = 该倍数 × 气体坐标跨度 |
| `ngrid_map` | 128 | 投影网格边长 |
| `vol_fine_grid` | 0.0 | 论文 App. A.2 的第二套精细 UV 网格；**默认关** |
| `map_clip_outside` | `False` | `True` = 投影盒外的气体夹到边界像素（质量守恒） |
| `k13_variant` | `'2p'` | 投影版 K13 里 n_CNM 的取法 |
| `rho_dm_mode` | `'none'` | 投影版 K13 的暗物质中面密度 |

几个值得知道的：

- **`map_clip_outside`**：默认 `False` 时投影盒外的气体 HI 记为零。
  设 `True` 会把它们夹到边界像素，质量守恒但会堆在边缘。
- **`vol_fine_grid`**：论文说需要第二套精细网格，否则星系中心的 UV 被低估。
  但论文没说两套网格怎么合并，而且打开会**破坏与参考实现的逐位一致**，
  所以默认关着 —— 它是实验开关，不是 bug 修复。
- **`k13_variant`**：官方 K13_map 产物用的是 `'2p'`；`'hydro'`（论文正文的
  写法）因为正反馈会奔向 f_mol = 1，实测差三个数量级。三个变体都留着做对照。

In [ ]:
# ---- 换一个配置再跑，看看参数怎么影响结果 ----
print('K13_map 在不同 n_CNM 取法下：')
print(f'{"k13_variant":<14} {"M_HI [M☉]":>13} {"M_H2 [M☉]":>13}')
print('-' * 44)
for variant in ('2p', 'hydro', 'hydro_nostar'):
    c = cfg.with_(k13_variant=variant)
    r = compute_hi_mass(gas_raw, 'K13_map', c, stars=star_raw, sub=sub)
    print(f'{variant:<14} {r.m_hi_msun:>13.4e} {r.m_h2_msun:>13.4e}')
print("\n→ 官方产物对应的是 '2p'（默认）。'hydro' 是论文附录 C.4 的写法，")
print('  但静水压底会让迭代正反馈奔向 f_mol=1，与官方产物差几个数量级。')

# ---- 质量守恒审计：投影盒内外 ----
r_map = results['K13_map']
d = r_map.diagnostics
print(f'\n投影几何（K13_map）：')
print(f'  投影盒边长 L = {d["L_map_kpc"]:.1f} kpc，像素 {d["pix_kpc"]*1e3:.0f} pc')
print(f'  盒内单元 {d["n_cells_kept"]} / {d["n_cells_total"]}')
print(f'  盒内气体质量占比 f_gas_in_map = {d["f_gas_in_map"]:.4f}')
print(f'  对齐成功 = {bool(d["align_success"])}（分支：{d["align_branch"]}，'
      f'用 {d["n_gas_align"]} 个气体单元 / {d["n_star_align"]} 个恒星粒子）')
if 'closure_m_h2' in d:
    print(f'  逐单元回加 vs 图积分的一致性：H2 {d["closure_m_h2"]:.6f}，'
          f'HI {d["closure_m_hi"]:.6f}   （1.0 = 完全自洽）')
print('\n→ 对齐向量（面向观测者）:', np.round(d['align_ang_mom_vector'], 4))

### 4.4 UV 场：`uv_mode`（只对体积版有意义）

体积版的 UV 场有两种取法：

| `uv_mode` | 含义 |
|---|---|
| `'adaptive_fft'`（默认） | 把恒星形成率的分布做 FFT 卷积，得到逐星系的 UV 场，再以 UVB 为下限 |
| `'global'` | 只用 Faucher-Giguère+2011 的均匀紫外背景 |

两者都**以背景为下限**，所以 `'global'` 就是"这个星系完全不造星"时的极限。

**投影版不吃这个参数** —— 它们的 UV 场是从 SFR 面密度图上做 2D 卷积来的，
那是模型定义的一部分。给 `*_map` 传 `uv_mode='global'` 会直接报错，
而不是默默忽略。

In [ ]:
if source.is_projected:
    print(f'{HI_MODEL} 是投影模型，uv_mode 不适用 —— 传 global 会报错：')
    try:
        TNGSource(basePath=basePath, snapNum=snapNum, subID=subID,
                  hi_model=HI_MODEL, uv_mode='global')
    except ValueError as e:
        print(f'  ValueError: {e}')
else:
    r_fft = compute_hi_mass(gas_raw, HI_MODEL, cfg, uv_mode='adaptive_fft')
    r_uvb = compute_hi_mass(gas_raw, HI_MODEL, cfg, uv_mode='global')
    print(f'{HI_MODEL} 在两种 UV 取法下：')
    print(f'  {"uv_mode":<14} {"M_HI [M☉]":>13} {"M_H2 [M☉]":>13} {"U_MW 峰值":>11}')
    print('  ' + '-' * 56)
    for name, r in (('adaptive_fft', r_fft), ('global', r_uvb)):
        print(f'  {name:<14} {r.m_hi_msun:>13.4e} {r.m_h2_msun:>13.4e} '
              f'{r.u_mw.max():>11.4g}')
    print(f'\n  HI 差 {100*(r_fft.m_hi_msun/r_uvb.m_hi_msun - 1):+.1f}%，'
          f'H2 差 {100*(r_fft.m_h2_msun/r_uvb.m_h2_msun - 1):+.1f}%')
    print('  → 恒星形成越活跃，这个差别越大。这是 v0.3.0 修掉的那个 bug：')
    print('    旧代码的插值器三条轴共用一套网格，adaptive_fft 实际上退化成 global。')

---

## 5. 构建合成观测

到这里 `source` 已经准备好了。接下来定义**观测**：望远镜看到的是什么网格、
什么波束、什么噪声。

```
DataCube（网格 + WCS）  +  Beam（波束）  +  Noise（噪声）
        +  SpectralModel（谱线形状）  +  SPHKernel（粒子怎么铺到像素上）
                              │
                              ▼
                          Martini
```

每个组件都是可替换的：换掉 `GaussianBeam` 就是另一台望远镜，
换掉 `CubicSplineKernel` 就是另一种沉积方案。

### 5.1 `DataCube`：像素、通道、谱轴单位

```python
DataCube(n_px_x, n_px_y, n_channels,
         px_size, channel_width, spectral_centre,
         ra, dec, spectral_unit=None)
```

三个量要按源的尺度来定，不能拍脑袋：

| 参数 | 怎么定 | 本教程 |
|---|---|---|
| `px_size` | 每个波束 FWHM 采 3 个像素以上（Nyquist） | 波束/3 |
| `n_px_x/y` | 视场 ≥ 源的角尺度（否则粒子被剔除，见 5.4） | 1.1× 源范围 |
| `channel_width` | 通道范围要盖住 **粒子速度范围 + 谱线半宽 ±4σ**（见 5.4），再切成几十个通道 | 见下一格的计算 |

### ⚠️ `spectral_unit`：v0.2.0 修掉的一个静默 bug

`spectral_unit=None`（默认）表示**沿用 `channel_width` 的单位**，
也就是 `20*U.km/U.s` 给速度轴、`100*U.kHz` 给频率轴。

如果显式传 `'MHz'`，`channel_width` 仍用速度单位时会被**差分**换算成频率。
旧版本这里用了 astropy 的 `doppler_radio` 等价关系 —— 那是个把**绝对**速度
映到**绝对**频率的映射，对"宽度"用会返回 `f_rest` 本身而不是一个小量，
结果立方体只剩 0.007% 的流量。**MHz 和 km/s 现在给出完全相同的流量。**

这一条会在本教程最后一节验证 —— 那里才跑，是因为验证用的小立方体会**就地裁剪
`source`**，而这件事本身值得单独讲（见第 8 节）。

In [ ]:
# ===== 先定几何：视场、像素、通道 =====
BEAM_FWHM   = 10.0 * U.arcsec
NOISE_RMS   = 0.5e-3 * U.Jy / U.beam     # 0.5 mJy/beam

# 像素 = 波束/3（每波束 FWHM 至少 3 像素）
px_size = (BEAM_FWHM / 3.0).to(U.arcsec)

# 视场：1.1× 源范围，但至少 3 倍波束 FWHM
fov_y = max(y_extent_arcsec, 3 * BEAM_FWHM.to_value(U.arcsec))
fov_z = max(z_extent_arcsec, 3 * BEAM_FWHM.to_value(U.arcsec))

n_px_x = int(np.ceil(fov_y / px_size.value));  n_px_x += n_px_x % 2
n_px_y = int(np.ceil(fov_z / px_size.value));  n_px_y += n_px_y % 2

# 只为估算谱线宽度而临时建一个；**不要**把它复用给多个 Martini（见第 8 节）
_sm_for_width = GaussianSpectrum(sigma='thermal')

# 谱轴：粒子速度范围 **加上** 谱线本身的宽度。
# Martini 在插入前会剔除"落不进立方体"的粒子，判据之一是每个粒子的
# 谱线 ±4σ 都要在通道范围内（σ 由 GaussianSpectrum 给）。热展宽 σ 可以到
# 100 km/s 量级（几千万 K 的气体），所以谱轴要比气体的运动学范围宽得多。
hw_max = float(np.max(_sm_for_width.half_width(source)).to_value(U.km / U.s))
v_span_needed = v_kms + 8.0 * hw_max            # ±4σ 两侧

channel_width = max(v_span_needed / 40.0, 5.0) * U.km / U.s
# +2 个通道的余量：取整可能让总宽刚好差一点点，而被剔掉的正是边缘粒子
n_channels = int(np.ceil(v_span_needed / channel_width.value)) + 2
n_channels += n_channels % 2

print(f'波束 FWHM   {BEAM_FWHM}')
print(f'像素        {px_size:.2f}  ({BEAM_FWHM/px_size:.1f} 像素 / 波束)')
print(f'视场        {n_px_x}×{n_px_y} px = {n_px_x*px_size.value:.0f}" × {n_px_y*px_size.value:.0f}"')
print(f'            源的角尺度 {y_extent_arcsec:.0f}" × {z_extent_arcsec:.0f}"')
print(f'谱轴        粒子视向展宽 {v_kms:.0f} km/s')
print(f'            最大热展宽半宽 σ = {hw_max:.0f} km/s  →  需要 ±4σ = ±{4*hw_max:.0f} km/s')
print(f'            合计 {v_span_needed:.0f} km/s  →  {n_channels} 通道 × {channel_width:.1f}')
print(f'噪声        {NOISE_RMS.to(U.mJy/U.beam):.1f}')


### 5.2 波束、噪声、谱线模型、SPH 核

| 组件 | 本教程用的 | 说明 |
|---|---|---|
| `GaussianBeam(bmaj, bmin, bpa, truncate)` | 10″ 圆波束 | `truncate=4` 即核截断在 4σ；`bmin` 默认等于 `bmaj` |
| `GaussianNoise(rms, seed)` | 0.5 mJy/beam | 给定 `seed` 结果可复现 |
| `GaussianSpectrum(sigma)` | `'thermal'` | 用粒子温度定谱线宽度；也可以给固定值如 `7*U.km/U.s` |
| `CubicSplineKernel()` | 三次样条 | MARTINI 的 SPH 核；像素太粗时会自动退化成高斯核 |

`GaussianSpectrum(sigma='thermal')` 是**物理上的正确选择**：21 cm 线的宽度
来自气体温度（热展宽），用模拟里每个粒子的温度算，比给一个固定值更有依据。

### 5.3 跑完整条管线

```python
M = Martini(source=source, datacube=dc, beam=beam, noise=noise,
            spectral_model=..., sph_kernel=...)
M.insert_source_in_cube(ncpu=4)   # 铺谱线，按像素并行
M.add_noise()                     # 可选
M.convolve_beam()                 # 可选
M.write_fits('out.fits')          # 或 write_hdf5
```

三步的顺序不能换：**先铺源、再加噪声、最后卷波束**。
先加噪声再卷波束会让噪声也因此被平滑，不再是"每波束"的噪声。

下一格的代码在铺完源之后、加噪声之前**多写了一份没加噪声的立方体**，
专门用来验质量守恒（见 5.4）—— 这个星系加了噪声之后单通道信噪比远小于 1，
在带噪声的立方体上求和会被噪声主导。注意 `write_fits()` 会去掉卷积用的 padding，
所以写完要补回来，否则下一步 `convolve_beam()` 会报错。

In [ ]:
datacube = DataCube(
    n_px_x=n_px_x, n_px_y=n_px_y, n_channels=n_channels,
    px_size=px_size,
    channel_width=channel_width,
    spectral_centre=source.vsys,
    ra=source.ra, dec=source.dec,
    spectral_unit='MHz',          # 频率轴；换成 None 就是速度轴（见 5.1）
)

beam  = GaussianBeam(bmaj=BEAM_FWHM, bmin=BEAM_FWHM, bpa=0*U.deg, truncate=4.0)
noise = GaussianNoise(rms=NOISE_RMS, seed=42)

M = Martini(source=source, datacube=datacube, beam=beam, noise=noise,
            spectral_model=GaussianSpectrum(sigma='thermal'),   # 每个 Martini 一个新实例
            sph_kernel=CubicSplineKernel())

print('铺源 ...')
M.insert_source_in_cube(progressbar=False, ncpu=4)   # 这一行会打印质量回收

# 先存一份**未加噪声、未卷积**的立方体。
# 质量守恒只能在这一份上验：加了噪声之后，单通道信噪比可能远小于 1
# （这个星系就是这么个情况），总和会被噪声主导。
fits_clean = os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_10arcsec_nonoise.fits')
M.write_fits(fits_clean)
print(f'已保存（干净）{fits_clean}')

# write_fits() 会调 datacube.drop_pad() 去掉卷积用的边缘填充，
# 所以写完之后要补回来，否则 next 一步 convolve_beam() 会报
# "datacube padding insufficient"。
# （填充是为了让波束卷积在图像边缘不把流量从对面绕回来。）
M.datacube.add_pad(M.beam.needs_pad())

print('\n加噪声 ...')
M.add_noise()
print('卷波束 ...')
M.convolve_beam()

fits_hi = os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_10arcsec.fits')
M.write_fits(fits_hi)
print(f'已保存 {fits_hi}')

### 5.4 验证：立方体里的质量对不对

上面 `insert_source_in_cube()` 打印的 `[xx% of initial source mass]` 是最重要
的一行诊断。**它应该是 100%**。

`Martini` 在插入之前会剔除"不可能贡献到立方体"的粒子
（`Martini._prune_particles`），判据有三条，全部开：

| 判据 | 含义 |
|---|---|
| **空间** | 粒子（连同它的 SPH 核尺寸）落在视场像素范围之外 |
| **谱轴** | 粒子的谱线（速度 **±4σ**，σ 由 `GaussianSpectrum` 给）落在通道范围之外 |
| **质量** | `mHI = 0` |

所以"视场够大"这句话对谱轴同样成立，而且**谱轴那一条比直觉宽得多**：
几千万 K 的气体热展宽 σ 能到 100 km/s 量级，它的谱线就要占 ±400 km/s 以上。
上一步 `v_span_needed = v_km + 8σ` 就是按这条判据算的。

如果回收率不到 100%，看打印的 `Pruned particles that will not contribute to
data cube, N particles remaining with total HI mass of ...` —— 总量掉了多少，
就是被剔除掉多少。宽一点的视场 / 通道范围即可。

下面这一格**不依赖 astrovodka 的任何内部变量**，直接从写出的 FITS 反算质量 ——
这正是观测者处理真实数据立方体的做法。（用的是上面那份**还没卷波束**的立方体，
单位是 `Jy/arcsec²`，换算最直接；如果立方体已经卷过波束、单位变成 `Jy/beam`，
还要再除以波束立体角 `π/(4 ln2)·bmaj·bmin`。）用标准公式：

$$M_{\rm HI}\ [M_\odot] = 2.36\times10^5 \cdot D_L^2 [\mathrm{Mpc}] \cdot \sum_{\rm chan} S_{\rm tot}\, \Delta v\ [\mathrm{Jy\cdot km/s}]$$

其中 $S_{\rm tot}$ 是该通道的**总**流量密度：把每个像素的
`Jy/arcsec²` 乘上像素面积再求和。单位全部从 FITS 头里读。

In [ ]:
with fits.open(fits_clean) as hdul:
    head = hdul[0].header
    cube = hdul[0].data.astype('float64')

px_arcsec = abs(head['CDELT1']) * 3600.0            # 度 -> 角秒（CDELT1 是负的）

# 通道宽：谱轴单位从 CUNIT3 读，不要假设
if head['CUNIT3'].strip().replace(' ', '') in ('ms-1', 'm/s'):
    dv_kms = abs(head['CDELT3']) / 1000.0
else:                                                # 频率轴 Hz
    dv_kms = abs(head['CDELT3']) / head['RESTFRQ'] * 299792.458

print(f'FITS 头: BUNIT={head["BUNIT"]!r}  CTYPE3={head["CTYPE3"]}  '
      f'CUNIT3={head["CUNIT3"]!r}')
print(f'  像素 {px_arcsec:.3f}"   通道 {cube.shape[0]} × {dv_kms:.3f} km/s')

# 这一份还没卷波束，单位是 Jy/arcsec^2，换算最直接：
# 每个通道的总流量密度 [Jy] = Σ_pix S[Jy/arcsec^2] × 像素面积
s_tot = cube.sum(axis=(1, 2)) * px_arcsec**2          # Jy
integrated = float(np.sum(s_tot * dv_kms))            # Jy km/s

DL_mpc = luminosity_distance(Z_OBS, cosmo).value
m_hi_fits = 2.36e5 * DL_mpc**2 * integrated           # Msun

print(f'\n积分流量 ∫S dv = {integrated:.6e} Jy km/s')
print(f'M_HI（从 FITS 反算）  = {m_hi_fits:.4e}')
print(f'M_HI（模型直接给的）  = {source.m_HI_total:.4e}')
ratio = m_hi_fits / source.m_HI_total.value
print(f'比值 = {ratio:.4f}')
print('\n→ 应当在 1.00 附近。')
print('  差的那不到 1%：边缘粒子的 SPH 核有一部分铺到了网格外，这部分流量')
print('  确实不在了 —— 这是有限视场的固有损失，不是 bug。')
print('  明显低于 ~0.99 才说明有整颗粒子被剔除，那就加大视场或通道范围。')
assert 0.99 < ratio < 1.01, f'质量回收异常：{ratio}'

# 如果立方体已经卷过波束（单位变成 Jy/beam），要再除以波束立体角：
#   S_total[Jy] = Σ_pix S[Jy/beam] × px_area / beam_area
#   beam_area = pi/(4 ln2) × bmaj × bmin
# 这里没卷，所以用不上；但读别人给的立方体时要注意 BUNIT。

# ---- 噪声是否如约 ----
with fits.open(fits_hi) as hdul:
    noisy = hdul[0].data.astype('float64')
rms_meas = float(np.std(noisy[0]))
print(f'\n噪声：请求 {NOISE_RMS.to(U.mJy/U.beam):.2f} mJy/beam，'
      f'最终立方体单通道实测 {rms_meas*1e3:.2f} mJy/beam')
peak_chan = float(np.max(noisy[0]))
print(f'  单通道峰值 {peak_chan*1e3:.2f} mJy/beam  →  单通道信噪比 '
      f'{peak_chan/rms_meas:.2f}')
print('  （这个星系在这个噪声下按通道看是探不到的，要在 moment 0 上积分才看得见）')

### 5.5 换一个角分辨率

同一个源、同一个噪声，只把波束从 10″ 换成 3′。
这是"高分辨率干涉阵" vs "单天线"的对比：低分辨率下每个波束装下更多流量，
单通道信噪比更高，但空间信息被抹平了。

In [ ]:
BEAM_B = 3.0 * U.arcmin
px_size_b = (BEAM_B / 3.0).to(U.arcsec)

fov_y_b = max(y_extent_arcsec, 3 * BEAM_B.to_value(U.arcsec))
fov_z_b = max(z_extent_arcsec, 3 * BEAM_B.to_value(U.arcsec))
n_px_x_b = int(np.ceil(fov_y_b / px_size_b.value));  n_px_x_b += n_px_x_b % 2
n_px_y_b = int(np.ceil(fov_z_b / px_size_b.value));  n_px_y_b += n_px_y_b % 2

datacube_b = DataCube(
    n_px_x=n_px_x_b, n_px_y=n_px_y_b, n_channels=n_channels,
    px_size=px_size_b, channel_width=channel_width,
    spectral_centre=source.vsys, ra=source.ra, dec=source.dec,
    spectral_unit='MHz',
)
print(f'波束 {BEAM_B:.0f}，像素 {px_size_b:.0f}"，'
      f'视场 {n_px_x_b}×{n_px_y_b} px = {n_px_x_b*px_size_b.value:.0f}" × '
      f'{n_px_y_b*px_size_b.value:.0f}"')

M_b = Martini(source=source, datacube=datacube_b,
              beam=GaussianBeam(bmaj=BEAM_B, bmin=BEAM_B, bpa=0*U.deg, truncate=4.0),
              noise=GaussianNoise(rms=NOISE_RMS, seed=42),
              spectral_model=GaussianSpectrum(sigma='thermal'),
              sph_kernel=CubicSplineKernel())

M_b.insert_source_in_cube(progressbar=False, ncpu=4)
M_b.add_noise()
M_b.convolve_beam()

fits_lo = os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_3arcmin.fits')
M_b.write_fits(fits_lo)
print(f'\n已保存 {fits_lo}')

---

## 6. 三维数据产品

`SubhaloDataProducts` 不是数据立方体，而是**三维网格** —— 适合做统计、
功率谱、或者喂给别的射线转移代码。

| 网格 | 单位 | 含义 |
|---|---|---|
| `uv_field` | Draine | UV 辐射场 |
| `dm_real` / `dm_rsd` | M☉/(ckpc/h)³ | 暗物质密度（实空间 / 红移空间） |
| `hi_real` / `hi_rsd` | M☉/(ckpc/h)³ | HI 密度 |
| `h2_real` / `h2_rsd` | M☉/(ckpc/h)³ | H₂ 密度 |

红移空间的坐标移动用 `s_x = x + v_los/(a·H(a))`，这是观测者实际会看到的。

> **两套坐标，别混。** 所有质量网格用的是**共动 ckpc/h**、
> 以 DM 质心为中心的盒子（`dp.grid_edges`）。
> 但 `uv_field` 在**物理 kpc** 的自适应盒里（`dp.metadata['uv_grid_bounds_physical_kpc']`）——
> 它跟着气体分布走。这两套不一样。

**6.3 节走到 group 尺度**：`GroupDataProducts` 把这颗子晕所属的整个 FOF group 网格化，
并把气体分成 ISM、CGM、IGM。

In [ ]:
print(f'计算 3D 数据产品（ngrid=128，模型 {HI_MODEL}）...')
dp = SubhaloDataProducts.from_subhalo(
    basePath=basePath, snapNum=snapNum, subID=subID,
    ngrid=128, hi_model=HI_MODEL,
    load_subhalo_only=True, cosmology=cosmo,
)

h5_path = os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_data_products.hdf5')
dp.save_hdf5(h5_path)
print(f'已保存 {h5_path}')

print(f'\n网格 {dp.dm_real.shape}，单位 {dp.metadata["grid_unit"]}')
print(f'  M_HI 总量 {dp.metadata["m_HI_Msun"]:.4e}  （应等于 source.m_HI_total）')
print(f'  M_H2 总量 {dp.metadata["m_H2_Msun"]:.4e}')
print(f'  用的模型  {dp.metadata["hi_model"]}')
print(f'  UV 场     {"无（投影模型）" if dp.uv_field is None else dp.uv_field.shape}')
if dp.metadata.get('uv_grid_bounds_physical_kpc'):
    b = dp.metadata['uv_grid_bounds_physical_kpc']
    print(f'    UV 盒 [物理 kpc] x=[{b[0]:.0f},{b[1]:.0f}] y=[{b[2]:.0f},{b[3]:.0f}] z=[{b[4]:.0f},{b[5]:.0f}]')

# ---- 往返测试 ----
dp2 = SubhaloDataProducts.load_hdf5(h5_path)
assert np.array_equal(dp.hi_real, dp2.hi_real)
print('\n✓ HDF5 往返一致')

### 6.1 密度投影

In [ ]:
def plot_projections(dp, fields, titles, cmaps, figname):
    """把 3D 网格沿三个轴投影出来，坐标轴用物理尺度标注。"""
    nrows = len(fields)
    fig, axes = plt.subplots(nrows, 3, figsize=(15, 4.3 * nrows))
    axes = np.atleast_2d(axes)
    ext = [dp.grid_edges[0][0], dp.grid_edges[0][-1],
           dp.grid_edges[1][0], dp.grid_edges[1][-1]]
    for r, (f, t, cm) in enumerate(zip(fields, titles, cmaps)):
        arr = getattr(dp, f)
        for c, lbl in enumerate(['xy', 'xz', 'yz']):
            proj = arr.sum(axis=2 - c)
            ax = axes[r, c]
            pos = proj[proj > 0]
            im = ax.imshow(proj.T, origin='lower', extent=ext, cmap=cm,
                           norm=LogNorm(vmin=pos.min() if pos.size else 1,
                                        vmax=proj.max()))
            ax.set_title(f'{t} — {lbl}', fontsize=11)
            ax.set_xlabel('ckpc/h'); ax.set_ylabel('ckpc/h')
            plt.colorbar(im, ax=ax, shrink=.85)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, figname), bbox_inches='tight', facecolor='white')
    plt.show()


plot_projections(
    dp,
    ['dm_real', 'hi_real', 'h2_real'],
    ['DM 密度', 'HI 密度', r'H$_2$ 密度'],
    ['bone', 'inferno', 'magma'],
    f'sub{subID}_z{Z_OBS}_DM_HI_projections.png',
)

### 6.2 UV 场与红移空间畸变

`dm_rsd − dm_real` 就是红移空间畸变（RSD）的效应 ——
本动速度把结构沿视线方向拉长（凯泽效应）或压扁（上帝手指）。

In [ ]:
def plot_uv_and_rsd(dp, figname):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
    ex, ey, ez = dp.grid_edges

    if dp.uv_field is not None:
        ax = axes[0]
        b = dp.metadata.get('uv_grid_bounds_physical_kpc')
        ext = [b[0], b[1], b[2], b[3]] if b else None
        im = ax.imshow(dp.uv_field.sum(axis=2).T, origin='lower', extent=ext,
                       cmap='afmhot', norm=LogNorm())
        ax.set_title('UV 场（沿 z 求和，物理 kpc）')
        ax.set_xlabel('kpc'); ax.set_ylabel('kpc')
        # 注意：这里画的是沿一条视线的**求和**，不是单个单元的 U_MW
        plt.colorbar(im, ax=ax, label=r'$\sum$ U_MW [Draine]', shrink=.85)
    else:
        axes[0].text(.5, .5, '投影模型不产生 3D UV 场网格\n（其 UV 场在投影平面上）',
                     ha='center', va='center', transform=axes[0].transAxes)
        axes[0].set_axis_off()

    ax = axes[1]
    diff = dp.dm_rsd - dp.dm_real
    v = np.percentile(np.abs(diff[diff != 0]), 99) if np.any(diff != 0) else 1
    im = ax.imshow(diff.sum(axis=2).T, origin='lower',
                   extent=[ex[0], ex[-1], ey[0], ey[-1]],
                   cmap='RdBu_r', vmin=-v, vmax=v)
    ax.set_title('红移空间畸变  DM_RSD − DM_real')
    ax.set_xlabel('ckpc/h'); ax.set_ylabel('ckpc/h')
    plt.colorbar(im, ax=ax, shrink=.85)

    ax = axes[2]
    k = dp.hi_real.shape[2] // 2
    im = ax.imshow(dp.hi_real[:, :, k].T, origin='lower',
                   extent=[ex[0], ex[-1], ey[0], ey[-1]],
                   cmap='inferno', norm=LogNorm())
    ax.set_title(f'HI 密度中面切片 (z 索引 {k})')
    ax.set_xlabel('ckpc/h'); ax.set_ylabel('ckpc/h')
    plt.colorbar(im, ax=ax, shrink=.85)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, figname), bbox_inches='tight', facecolor='white')
    plt.show()


plot_uv_and_rsd(dp, f'sub{subID}_z{Z_OBS}_UV_and_RSD.png')

### 6.3 把镜头拉远：FOF group

每个 subhalo 都知道自己属于哪个 group——`SubhaloGrNr`。我们这颗运气不太好：

```python
loadSingle(basePath, snapNum, subhaloID=subID)['SubhaloGrNr']    # -> 0
```

Group 0 是整个盒子里最大的天体：R200c = 1.5 Mpc 之内有 **17185 个 subhalo、4.5e7 个气体
胞元**。光读进来就是好几 GB，所以这一节换一个小一些的 group——方法完全一样。

`GroupDataProducts.from_group()` 把整个 **FOF group**（它包含的每个 subhalo，加上只属于
group 本身的那部分）网格化，并按气体的实际位置分成三类：

| 类别 | 定义 |
|---|---|
| **ISM** | 归属于某个 subhalo，且在其 `min(k_gas · R_half,gas, k_star · R_half,star)` 之内 |
| **CGM** | 归属于某个 subhalo，但在该半径之外 |
| **IGM** | 在 group 内但**不属于任何** subhalo——即 group 的 fuzz |

归属关系不是空间上的猜测：catalog 为每个 subhalo 记录了它在快照粒子数组中拥有的气体
胞元区间（`Subhalo/SnapByType` + `SubhaloLenType`），所以这个划分是精确的，而且不需要
额外读取。

把这套东西用到自己的 group 上之前，有三点值得知道：

- **ISM 半径必须取两个半径的较小值。** group 的中心 subhalo 包含整个热晕，它的*气体*
  半质量半径约 260 ckpc/h，而*恒星*半质量半径只有约 19 ckpc/h（group 59）。只用气体
  半径、取 `k=0.5`，就已经会把 group 15% 的质量划成 ISM。
- **HI 是逐 subhalo 计算的**，每个都在自己的自适应 FFT 盒里——这正是这些模型被标定时
  所用的方案。对整个 group 只做一次会把 UV 场抹到约 30 kpc 的格子上。只有含气体的
  subhalo 需要计算：group 59 的 1710 个成员里只有 68 个。
- **各类占比与 `ngrid` 无关**——它们是逐胞元的量；网格只决定结果怎么展示。

In [ ]:
# 我们这颗 subhalo 究竟住在哪里？它的 group 又要多大代价？
# 各读一行 catalog 而已——很便宜，先看清楚再决定要不要载入。
h = loadHeader(basePath, snapNum)['HubbleParam']
for gid, label in ((int(loadSingle(basePath, snapNum, subhaloID=subID)['SubhaloGrNr']),
                    '我们这颗 subhalo 所属的 group'), (GROUP_ID, '下面实际使用的 group')):
    g = loadSingle(basePath, snapNum, haloID=gid)
    print(f"{label:<22} group {gid:>5}: {int(g['GroupNsubs']):>6} subhalos, "
          f"{int(g['GroupLenType'][0]):>12,} gas cells, "
          f"{int(g['GroupLenType'][1]):>12,} DM particles, "
          f"M200c = {g['Group_M_Crit200'] * 1e10 / h:.2e} Msun")

import time
t0 = time.time()
gdp = GroupDataProducts.from_group(
    basePath, snapNum, GROUP_ID,
    ngrid=GROUP_NGRID,
    hi_model=HI_MODEL,          # 只有体密度模型适用于整个 group
    ncpu=1,
)
print(f"\n载入 group {GROUP_ID} 用了 {time.time() - t0:.0f} 秒")

md = gdp.metadata
print(f"{md['Nsubs']} 个 subhalo（其中 {md['N_sub_with_gas']} 个含气体），"
      f"{md['n_gas']:,} 个气体胞元，R200c = {md['R200c_kpc']:.0f} kpc，"
      f"M200c = {md['M200c_msun']:.2e} Msun")
print(f"盒子半宽 [视线, y, z] = "
      f"{[round(float(v), 1) for v in md['box_halfsize_ckpc']]} ckpc/h")

print(f"\n{'类别':<6}{'胞元数':>11}{'气体质量 [Msun]':>18}{'HI 质量 [Msun]':>17}"
      f"{'f_HI':>8}{'气体占比':>10}")
for c in ('ism', 'cgm', 'igm'):
    gas, hi = md[f'm_gas_{c}_msun'], md[f'm_hi_{c}_msun']
    print(f"{c.upper():<6}{md['n_cells_' + c]:>11,}{gas:>18.4e}{hi:>17.4e}"
          f"{hi / max(gas, 1):>8.4f}{100 * gas / md['m_gas_total_msun']:>9.2f}%")

# 三类恰好把 group 分完，这是构造上就保证的……
assert md['n_cells_ism'] + md['n_cells_cgm'] + md['n_cells_igm'] == md['n_gas']
assert abs((md['m_gas_ism_msun'] + md['m_gas_cgm_msun'] + md['m_gas_igm_msun'])
           / md['m_gas_total_msun'] - 1) < 1e-9
# ……而且这个分类是有物理含义的：原子气体就在 ISM 里。
f_hi = {c: md[f'm_hi_{c}_msun'] / md[f'm_gas_{c}_msun'] for c in ('ism', 'cgm', 'igm')}
assert f_hi['ism'] > f_hi['cgm'] > f_hi['igm'], f_hi
print(f"\n\u2713 三类恰好分完整个 group，且 f_HI(ISM)={f_hi['ism']:.3f} "
      f"> f_HI(CGM)={f_hi['cgm']:.3f} > f_HI(IGM)={f_hi['igm']:.4f}")
print(f"  fuzz 占气体质量的 {100 * md['m_gas_igm_msun'] / md['m_gas_total_msun']:.1f}%，"
      f"占暗物质的 {100 * md['m_dm_igm_total_msun'] / md['m_dm_total_msun']:.1f}%")

In [ ]:
def plot_group_classes(dp, figname):
    """三类气体在整个 group 中的分布，以及最亮星系的放大图。"""
    s = dp.metadata['hubble_h'] * dp.metadata['scale_factor']   # 1 ckpc/h 对应多少物理 kpc
    e = [np.asarray(x) / s for x in dp.grid_edges]
    centre = np.asarray(dp.metadata['group_pos_ckpc_h']) / s
    r_ism = ism_radius(dp.subhalos['rhalf_type'][:, 0], dp.subhalos['rhalf_type'][:, 4],
                       dp.metadata['ism_k_gas'], dp.metadata['ism_k_star'])

    fig, axes = plt.subplots(2, 3, figsize=(16, 8.8))

    # ---- 第一行：整个 group，按类别分开 ----
    for c, cls in enumerate(('ism', 'cgm', 'igm')):
        ax = axes[0, c]
        img = getattr(dp, f'gas_{cls}_real').sum(axis=2).T      # 行是 y，列是 x
        pos = img[img > 0]
        im = ax.imshow(img, origin='lower', aspect='auto',
                       extent=[e[0][0] - centre[0], e[0][-1] - centre[0],
                               e[1][0] - centre[1], e[1][-1] - centre[1]],
                       cmap='magma',
                       norm=LogNorm(vmin=pos.min() if pos.size else 1e-6,
                                    vmax=np.percentile(pos, 99.5) if pos.size else 1.0))
        ax.add_patch(Circle((0, 0), dp.metadata['R200c_ckpc_h'] / s, fill=False,
                            ec='cyan', ls='--', lw=1.2, label='R200c'))
        frac = 100 * dp.metadata[f'm_gas_{cls}_msun'] / dp.metadata['m_gas_total_msun']
        ax.set_title(f'{cls.upper()} — 占气体 {frac:.1f}%')
        ax.set_xlabel(r'$\Delta$x [kpc]'); ax.set_ylabel(r'$\Delta$y [kpc]')
        plt.colorbar(im, ax=ax, shrink=.85, label=r'$M_\odot$/kpc$^3$')

    # ---- 第二行：最亮星系，用逐胞元数据重新分箱 ----
    # 整组网格一格约 30 kpc，直接放大只会看到方块；
    # dp.cells 让我们能只对这一小块按星系尺度重新做直方图。
    xyz = np.asarray(dp.cells['coords_ckpc_h'], dtype='float64')
    mass = np.asarray(dp.cells['m_gas_msun'], dtype='float64')
    klass = np.asarray(dp.cells['class_id'])
    gal = int(np.argmax(np.asarray(dp.subhalos['mass_type'])[:, 0]))   # 气体最多者
    gc = np.asarray(dp.subhalos['pos'])[gal]
    half = float(np.clip(4.0 * r_ism[gal], 60.0 * s, 0.3 * dp.metadata['R200c_ckpc_h']))
    edges = [np.linspace(gc[i] - half, gc[i] + half, 257) for i in range(2)]
    inside = np.all(np.abs(xyz - gc) < half, axis=1)
    area = (edges[0][1] - edges[0][0]) * (edges[1][1] - edges[1][0])
    ext = [edges[0][0] / s - gc[0] / s, edges[0][-1] / s - gc[0] / s,
           edges[1][0] / s - gc[1] / s, edges[1][-1] / s - gc[1] / s]
    maps, ranges = {}, {}
    for cls, cid in (('ISM', 0), ('CGM', 1)):
        sel = inside & (klass == cid)
        maps[cls] = np.histogram2d(xyz[sel, 0], xyz[sel, 1], bins=edges,
                                   weights=mass[sel])[0] / area
        pos = maps[cls][maps[cls] > 0]
        ranges[cls] = (np.percentile(pos, 25) if pos.size else 1e-6,
                       np.percentile(pos, 99.5) if pos.size else 1.0)
    for c, key in enumerate(('ISM', 'CGM', None)):
        ax = axes[1, c]
        if key is None:
            img = np.zeros(maps['ISM'].T.shape + (3,))
            for cls, rgb in (('ISM', 0), ('CGM', 2)):
                lo, hi = ranges[cls]
                img[..., rgb] = np.clip(
                    (np.log10(np.maximum(maps[cls].T, lo)) - np.log10(lo))
                    / max(np.log10(hi) - np.log10(lo), 1e-9), 0, 1)
            ax.imshow(img, origin='lower', extent=ext)
            ax.set_title('两者叠加（红 = ISM，蓝 = CGM）')
        else:
            lo, hi = ranges[key]
            im = ax.imshow(maps[key].T, origin='lower', extent=ext, cmap='magma',
                           norm=LogNorm(vmin=lo, vmax=hi))
            ax.set_title(f'{key}：最亮星系')
            plt.colorbar(im, ax=ax, shrink=.85, label=r'$M_\odot$/kpc$^2$')
        ax.set_xlabel(r'$\Delta$x [kpc]'); ax.set_ylabel(r'$\Delta$y [kpc]')

    fig.suptitle(f"group {dp.metadata['groupID']}：ISM 在星系内部，CGM 在它们的晕里，"
                 f"IGM 则是没有任何束缚结构的地方（下排：±{half / s:.0f} kpc）")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, figname), bbox_inches='tight', facecolor='white')
    plt.show()


plot_group_classes(gdp, f'group{GROUP_ID:07d}_snap{snapNum:03d}_ism_cgm_igm.png')

---

## 7. 可视化数据立方体

一个方便的函数：从 FITS 读回立方体，画 moment 0/1、整体谱线和几个通道图。
这是检验"我到底做出了什么"的标准做法。

In [ ]:
def compute_moments(cube, snr=3.0):
    """Moment 0 和 moment 1。

    Moment 1 只在**有信号**的像素上算 —— 否则会拿噪声去除以接近零的 moment 0，
    得到几万 km/s 的垃圾值。没有信号的像素置 NaN。
    """
    mom0 = cube.sum(axis=0)
    noise = np.std(cube[0])                              # 单通道噪声
    # moment-0 的噪声按噪声传播估计：n 个独立通道求和 -> sqrt(n) 倍。
    # （不用图像边缘实测：这里的视场只有源的 1.1 倍，边上也有源，没有干净区域。）
    mom0_noise = noise * np.sqrt(cube.shape[0])
    sig = mom0 > snr * mom0_noise
    with np.errstate(invalid='ignore', divide='ignore'):
        mom1 = (cube * np.arange(cube.shape[0])[:, None, None]).sum(axis=0) / mom0
    mom1 = np.where(sig, mom1, np.nan)
    return mom0, mom1, noise, sig


def visualize_cube(fits_path, title, figname, n_chan_maps=5):
    with fits.open(fits_path) as hdul:
        cube = hdul[0].data.astype('float64')
        head = hdul[0].header
    mom0, mom1, noise, sig = compute_moments(cube)
    nchan = cube.shape[0]
    # 源的谱中心在 cube 中心（datacube 是以 source.vsys 为中心的）
    centre_chan = nchan // 2
    mom0_noise = noise * np.sqrt(nchan)
    peak_snr = mom0.max() / mom0_noise

    fig = plt.figure(figsize=(16, 8.6))
    gs_top = fig.add_gridspec(1, 3, top=.86, bottom=.42, wspace=.34)
    gs_bot = fig.add_gridspec(1, n_chan_maps, top=.32, bottom=.04, wspace=.28)
    fig.subplots_adjust(left=.06, right=.97)

    ax = fig.add_subplot(gs_top[0, 0])
    vmax = np.percentile(mom0[sig], 98) if sig.any() else mom0.max()
    im = ax.imshow(mom0, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    ax.set_title('Moment 0（积分流量）', fontweight='bold')
    ax.contour(sig, levels=[.5], colors='w', linewidths=.6, alpha=.6)
    plt.colorbar(im, ax=ax, label='Jy/beam · channel')

    ax = fig.add_subplot(gs_top[0, 1])
    if np.any(np.isfinite(mom1)):
        lo, hi = np.nanpercentile(mom1, [2, 98])
        im = ax.imshow(mom1, origin='lower', cmap='RdBu_r', vmin=lo, vmax=hi)
        plt.colorbar(im, ax=ax, label='通道序号')
    else:
        ax.text(.5, .5, '信号不足，无法计算速度场', ha='center', va='center',
                transform=ax.transAxes)
    ax.set_title('Moment 1（速度场，仅信号像素）', fontweight='bold')

    ax = fig.add_subplot(gs_top[0, 2])
    spec = cube.sum(axis=(1, 2))
    cnoise = noise * np.sqrt(cube.shape[1] * cube.shape[2])
    ax.plot(spec, np.arange(nchan), 'k-', lw=1.4)
    ax.axvspan(-cnoise, cnoise, color='C0', alpha=.18, label=r'±1σ（全图求和）')
    ax.axhline(centre_chan, color='r', ls='--', lw=1, label='谱中心')
    ax.invert_yaxis()
    ax.set_xlabel('总流量 [Jy/beam]'); ax.set_ylabel('通道')
    ax.set_title('整体谱线', fontweight='bold')
    ax.legend(fontsize=8)

    chans = np.unique(np.clip(
        np.linspace(centre_chan - n_chan_maps // 2,
                    centre_chan + n_chan_maps // 2, n_chan_maps),
        0, nchan - 1).astype(int))
    for i, ch in enumerate(chans):
        ax = fig.add_subplot(gs_bot[0, i])
        ax.imshow(cube[ch], origin='lower', cmap='inferno',
                  vmin=0, vmax=np.percentile(cube[ch], 99.5))
        ax.set_title(f'通道 {ch}', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title, fontsize=15, fontweight='bold', y=.96)
    plt.savefig(os.path.join(OUTDIR, figname), bbox_inches='tight', facecolor='white')
    plt.show()

    print(f'{os.path.basename(fits_path)}')
    print(f'  单通道噪声 {noise*1e3:.2f} mJy/beam')
    print(f'  moment-0 噪声（传播估计）{mom0_noise*1e3:.3f} mJy/beam·channel')
    print(f'  moment-0 峰值/噪声 = {peak_snr:.1f}')
    print(f'  >3σ 的像素 {int(sig.sum())} / {sig.size}')
    print(f'  注意：这里是**逐像素**信噪比。源是延展的，把它整个积分起来')
    print(f'  的显著性要高得多 —— 这就是要看 moment 0 而不是单通道图的原因。')


visualize_cube(fits_hi, f'TNG-100 sub{subID}, z={Z_OBS}, 10 arcsec beam',
               f'sub{subID}_z{Z_OBS}_10arcsec_viz.png')
visualize_cube(fits_lo, f"TNG-100 sub{subID}, z={Z_OBS}, 3 arcmin beam",
               f'sub{subID}_z{Z_OBS}_3arcmin_viz.png')

### 7.1 两种分辨率并排比

同样的流量，被不同大小的波束摊开。低分辨率下每个波束里的流量更高、
单通道信噪比更好，但结构被抹平 —— 这正是为什么 HI 观测要权衡。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, fpath, label in zip(axes, [fits_hi, fits_lo],
                            ['10 arcsec beam', "3 arcmin beam"]):
    with fits.open(fpath) as hdul:
        d = hdul[0].data.astype('float64')
    mom0 = d.sum(axis=0)
    rms = np.std(d[0])
    sig = mom0 > 3 * rms
    vmax = np.percentile(mom0[sig], 98) if sig.any() else mom0.max()
    im = ax.imshow(mom0, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    peak_sig = mom0.max() / np.std(mom0[~sig])
    ax.set_title(f'{label}   （moment-0 峰值/噪声 = {peak_sig:.1f}）',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('RA pixel'); ax.set_ylabel('Dec pixel')
    plt.colorbar(im, ax=ax, label='Jy/beam · channel', shrink=.85)
plt.suptitle(f'TNG-100 sub{subID} @ z={Z_OBS} — Moment 0', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_comparison.png'),
            bbox_inches='tight', facecolor='white')
plt.show()

---

## 8. 一个必须知道的陷阱：`Martini` 会**就地**裁剪 source

`Martini._prune_particles` 把"落不进立方体"的粒子从 **`source` 对象本身**删掉
（`source.apply_mask`），不是只在这次观测里忽略它们。后果是：

> **用同一个 `source` 连做两个不同大小的立方体，第二个会少一截。**

第一个立方体视场小 → 剪掉一圈粒子 → `source.npart` 变小 → 第二个立方体
拿到的已经是残缺的源。而且这个损失是**静默**的，只体现在
`insert_source_in_cube` 打印的回收率里。

正确做法：

- **一个 source 对应一次观测**；要做多组参数扫描，就每组重新构造 `TNGSource`；
- 或者确保第一个立方体的视场/谱轴已经装得下整个源（回收率 100%），
  这时没有粒子被剪，source 不受影响。

下面这个检查就演示了这件事：它在**所有科学产物都做完之后**才跑，
用的还是同一个 `source` —— 因为那三个小立方体一定会剪 source。

In [ ]:
# ===== 验证谱轴单位：km/s 与 MHz 必须给出同一个立方体 =====
# 放在最后！下面这几个小立方体视场装不下整个星系，会就地裁剪 source。
# 如果放在前面跑，后面所有立方体都会缺一截。
print('用 24×24×16 的小立方体比较两种谱轴（只为比流量，视场故意不够）\n')

flux = {}
for unit in ('km/s', 'MHz'):
    dc_small = DataCube(n_px_x=24, n_px_y=24, n_channels=16,
                        px_size=px_size, channel_width=channel_width,
                        spectral_centre=source.vsys, ra=source.ra, dec=source.dec,
                        spectral_unit=unit)
    m = Martini(source=source, datacube=dc_small, quiet=True,   # quiet 是构造参数
                spectral_model=GaussianSpectrum(sigma='thermal'),
                sph_kernel=CubicSplineKernel())
    m.insert_source_in_cube(progressbar=False)
    flux[unit] = float(np.sum(m.datacube._array.to_value(U.Jy / U.arcsec**2)))
    print(f'  spectral_unit={unit!r:6}  CTYPE3={dc_small.wcs.wcs.ctype[2]:>4}  '
          f'通道宽={dc_small.channel_width}  立方体总流量={flux[unit]:.6e}')

ratio = flux['MHz'] / flux['km/s']
print(f'\n  流量比 MHz/(km/s) = {ratio:.6f}   （1.000000 = 一致）')
assert abs(ratio - 1.0) < 1e-6, '谱轴单位换算错了！'
print('  ✓ 两种谱轴给出同一个立方体。')
print('    旧版本这里是 0.00007 —— 那时它把 channel_width 用 doppler_radio')
print('    换算，返回的是 f_rest 本身而不是宽度，立方体只剩噪声。')
print(f'\n  跑完之后 source.npart = {source.npart}（{source.n_gas} 个气体单元里被剪掉了这些）')

---

## 9. 疑难排解

### 报错

| 报错 | 原因 / 怎么办 |
|---|---|
| `ValueError: unknown hi_model 'K13_volumn'` | 名字拼错。合法值是 `GD14_vol`、`GD14_map`、`K13_vol`、`K13_map`（大小写不敏感）。校验在读盘之前，不会白等。 |
| `ValueError: uv_mode='global' does not apply to K13_map` | 投影模型的 UV 场来自 SFR 面密度图，是模型定义的一部分。去掉 `uv_mode`，或改用 `*_vol`。 |
| `ValueError: the K13_map model projects the galaxy, so it needs the stellar particle data` | 直接用 `compute_hi_mass` 时投影模型要传 `stars=` 和 `sub=`。用 `TNGSource` 不会遇到。 |
| `KeyError: 'Coordinates' ...` | 传给 `compute_hi_mass` 的 gas 字典缺字段。体积版需要 8 个字段，投影版不需要 `InternalEnergy` 但要恒星。 |
| `UserWarning: ... angular diameter distance of only 4.4 kpc` | snapshot 99 的 z≈0，没传 `observed_redshift`。见 3.2。 |
| `TypeError: unknown HIModelConfig field(s) ['n_grid']` | `hi_config` 里字段名写错，报错里会列出可用的名字。 |
| 第二个立方体的回收率莫名变低 | **`Martini` 会就地裁剪 `source`。** 前一个立方体视场/谱轴太小，把粒子从 source 里删掉了。见第 8 节：一个 source 一次观测，或保证第一个立方体就是 100%。 |
| `ValueError: datacube padding insufficient for beam convolution` | 在 `convolve_beam()` 之前调过 `write_fits()`/`write_hdf5()` —— 它们会 `drop_pad()`。要么先写完再卷积，要么写完补回来：`M.datacube.add_pad(M.beam.needs_pad())`。 |
| `ValueError: cannot reshape array of size ... into shape ...` | **同一个 `GaussianSpectrum` 实例被多个 `Martini` 复用了。** 谱数组缓存在模型对象上，`Martini` 只在它还是 `None` 时重新初始化，所以换了立方体形状（尤其是通道数）就会套用旧的形状。每个 `Martini` 建一个新实例，或显式调 `M.init_spectra()`。 |

### 结果看着不对

| 现象 | 多半是 |
|---|---|
| `insert_source_in_cube` 报的质量回收 << 100%，且打印 `Pruned particles ...` | **立方体视场太小**。把 `n_px_x/n_px_y` 按源的角尺度放大（5.1 节）。 |
| 立方体全是噪声 | 源本身 HI 太少（检查 `source.m_HI_total`），或噪声 `rms` 太大，或通道范围没盖住源的视向速度。 |
| 投影模型的总 HI 比体积版少 | 正常：投影盒外的气体记零。看 `source.hi_diagnostics['f_gas_in_map']`；低于 0.9 时 `TNGSource` 会警告。想保住就设 `hi_config={'map_clip_outside': True}`。 |
| 加载一个子晕要几分钟、内存爆掉 | **取到团核心了**。catalog 按质量降序，ID 0 有约 3800 万单元。改用 100~200 区间。 |
| `adaptive_fft` 和 `global` 结果几乎一样 | 这个星系不怎么造星，UV 场峰值贴着本底。换一个恒星形成活跃的子晕才看得出差别。 |

### 数值一致性检查清单

任何时候都值得跑一遍这三条：

1. `insert_source_in_cube()` 的回收率 = **100%**
2. 从 FITS 反算的 M_HI ≈ `source.m_HI_total`（5.4 节）
3. 两个体积版的 `m_neutral` **完全相同**（模型只在 HI/H₂ 怎么分上有别）

---

## 10. 小结与下一步

### 这一趟做了什么

1. **选子晕** —— 避开按质量降序排列的团核心
2. **加载源** —— `TNGSource` 读快照、算 Planck15 距离、跑选定 HI 模型
3. **对比四种模型** —— `compute_hi_mass(gas, model=...)`，同一份数据跑四遍
4. **建观测** —— `DataCube`（含谱轴单位陷阱）+ beam + noise + 谱线模型 + SPH 核
5. **验证** —— 质量回收 100%，FITS 反算与模型一致，两种谱轴流量相同
6. **三维产物** —— UV/DM/HI/H₂ 网格，实空间与红移空间
7. **把镜头拉远到 group** —— `GroupDataProducts` 用 catalog 自带的归属关系把整个
   FOF group 分成 ISM / CGM / IGM
8. **可视化** —— moment maps、谱线、通道图

### 关键接口速查

```python
# 四种模型，任选其一
TNGSource(..., hi_model='K13_vol')          # 高层：一步到位
compute_hi_mass(gas, 'K13_map', cfg, stars=stars, sub=sub)   # 低层：只算物理

# 配置
HIModelConfig(h=0.6774, z=0.0)              # 完整对象
HIModelConfig.build({'ngrid': 64}, h=..., z=...)   # 以快照为准地合并
TNGSource(..., hi_config={'ngrid': 64})     # dict 也行

# 模型结果
source.mHI_g, source.m_H2_g, source.f_mol_g, source.u_mw_g
source.hi_diagnostics['f_gas_in_map']

# 这颗子晕所属的整个 FOF group，按类别拆分
gdp = GroupDataProducts.from_group(basePath, snapNum, GROUP_ID, ngrid=192)
gdp.metadata['m_gas_ism_msun'], gdp.metadata['m_hi_total_msun']
gdp.gas_ism_real, gdp.cells['class_id']    # 网格，以及逐胞元的类别标签
```

### 文档与验证

| 想了解 | 看哪 |
|---|---|
| API 全貌 | [`docs/zh/api/`](../../docs/zh/api/index.md)（`himodel.md` 讲四种模型） |
| 用户指南 | [`docs/zh/guide.md`](../../docs/zh/guide.md) |
| 改动的来龙去脉、逐位验证记录 | [`docs/zh/verification.md`](../../docs/zh/verification.md) |
| 安装与环境配置 | [`docs/zh/installation.md`](../../docs/zh/installation.md) |
| 单元测试（不需要 TNG 数据） | `pytest tests/ -q` |

### 还可以做什么

- **换红移**：改 `Z_OBS` 就能看同一个星系在不同距离上的可观测性
- **换模型**：改 `HI_MODEL`，四种模型都走通同一条管线
- **多星系同框**：上游 MARTINI 的 `CombinedSource` 没摘进来；
  多源叠加需要在 `Martini` 之上自己拼
- **真实望远镜响应**：换掉 `GaussianBeam`，或给 `DataCube` 传已有观测的 WCS
  （`DataCube.from_wcs`）
- **换成你自己的 group**：改掉 `GROUP_ID`，或照 6.3 用 `SubhaloGrNr` 找到某颗子晕
  所属的 group；`examples/zh/group_cluster_viz.py` 还能一次出四张图

In [ ]:
print('=' * 66)
print('  astrovodka 教程 — 完成')
print('=' * 66)
print(f'  子晕   {subID}   （snapshot {snapNum}）')
print(f'  HI 模型  {HI_MODEL}')
print(f'  观测红移 z = {Z_OBS}   D_L = {DL:.1f}   D_A = {DA:.1f}')
print(f'  M_HI     {source.m_HI_total:.4e}')
print(f'  M_H2     {source.m_H2_g.sum():.4e}')
print()
print('  产物：')
for f in ['model_diagnostics.png', 'data_products.hdf5', 'DM_HI_projections.png',
          'UV_and_RSD.png', '10arcsec_nonoise.fits', '10arcsec.fits',
          '3arcmin.fits', '10arcsec_viz.png', '3arcmin_viz.png',
          'comparison.png']:
    print(f'    sub{subID}_z{Z_OBS}_{f}')
print(f'\n  全部在 {OUTDIR}')